# Day 25: Multimodal Search Engine (CLIP + FAISS)

Build a search engine that finds images using text or image queries.

In [ ]:
import torch
import faiss
import numpy as np
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import os
from pathlib import Path
import requests
from io import BytesIO
import matplotlib.pyplot as plt

In [ ]:
# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP loaded on", device)

## 1. Build a small image collection
We'll download a few sample images (or you can use a folder of your own).

In [ ]:
# Sample image URLs
image_urls = {
    "cat": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png",
    "dog": "https://images.dog.ceo/breeds/hound-afghan/n02088094_1003.jpg",
    "beach": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beach.png",
    "tree": "https://cdn.pixabay.com/photo/2015/04/23/22/00/tree-736885_1280.jpg",
    "car": "https://images.pexels.com/photos/170811/pexels-photo-170811.jpeg"
}

# Download and store images
image_dir = Path("search_images")
image_dir.mkdir(exist_ok=True)
image_paths = []

for name, url in image_urls.items():
    response = requests.get(url, stream=True)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    save_path = image_dir / f"{name}.jpg"
    img.save(save_path)
    image_paths.append(save_path)
    print(f"Saved {save_path}")

## 2. Generate embeddings for all images

In [ ]:
image_embeddings = []
for path in image_paths:
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs).cpu().numpy()
        image_embeddings.append(emb)

image_embeddings = np.vstack(image_embeddings).astype('float32')
print(f"Embeddings shape: {image_embeddings.shape}")

## 3. Build FAISS index

In [ ]:
dim = image_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product (cosine similarity after normalisation)
faiss.normalize_L2(image_embeddings)
index.add(image_embeddings)
print(f"Index contains {index.ntotal} images")

## 4. Text‑to‑image search

In [ ]:
def search_text(query_text, k=2):
    inputs = processor(text=[query_text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_emb = model.get_text_features(**inputs).cpu().numpy().astype('float32')
    faiss.normalize_L2(text_emb)
    scores, indices = index.search(text_emb, k)
    return indices[0], scores[0]

query = "a cute cat"
indices, scores = search_text(query)
print(f"Query: '{query}'")
for idx, score in zip(indices, scores):
    print(f"{image_paths[idx].name}: similarity {score:.4f}")
    display(Image.open(image_paths[idx]))

## 5. Image‑to‑image search
Use one image as the query to find similar images.

In [ ]:
def search_image(query_image_path, k=2):
    img = Image.open(query_image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        img_emb = model.get_image_features(**inputs).cpu().numpy().astype('float32')
    faiss.normalize_L2(img_emb)
    scores, indices = index.search(img_emb, k)
    return indices[0], scores[0]

# Use the dog image as query
dog_path = image_dir / "dog.jpg"
indices, scores = search_image(dog_path)
print("Query image:")
display(Image.open(dog_path))
print("Results:")
for idx, score in zip(indices, scores):
    print(f"{image_paths[idx].name}: similarity {score:.4f}")
    display(Image.open(image_paths[idx]))

## 6. Wrap into a simple interactive function

In [ ]:
def multimodal_search(query, k=3):
    if query.endswith(('.jpg', '.png', '.jpeg')) and Path(query).exists():
        print("Image query")
        indices, scores = search_image(query)
    else:
        print("Text query")
        indices, scores = search_text(query)
    for idx, score in zip(indices, scores):
        print(f"{image_paths[idx].name}: {score:.4f}")
        display(Image.open(image_paths[idx]))

# Example usage
multimodal_search("a car on the road")
# multimodal_search("search_images/cat.jpg")  # if you want image query